In [ ]:
#!/usr/bin/env python
# coding: utf-8

import math
from tqdm import tqdm
import nltk
from nltk.corpus import stopwords
import ssl
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context
nltk.download('stopwords')
from nltk.tokenize import word_tokenize
import pandas as pd
from datetime import datetime
import pickle
import re
from pyxdameraulevenshtein import damerau_levenshtein_distance
import apsw
import sys
import numpy as np
import corp_simplify_utils
import seaborn as sns
import matplotlib.pyplot as plt
import pyreadr
from collections import Counter

# nlp
import spacy
from spacy import displacy
from collections import Counter
# to install: $python3 -m spacy download en_core_web_lg
import en_core_web_lg

# analysis/regressions
import statsmodels.api as sm
from statsmodels.formula.api import glm
from statsmodels.genmod.families import Poisson
from scipy.stats import ks_2samp
from scipy.stats import mannwhitneyu
from scipy.stats import ttest_ind

# from statsmodels.graphics.gofplots import qqplot_2samples
from scipy import stats
from joypy import joyplot
from matplotlib import cm

from datetime import date
today_for_filenames = date.today()
curr_date = str(today_for_filenames.strftime("%Y%m%d"))


NUMBER_OF_MATCHES_TO_RECORD = 10
punc_remove_re = re.compile(r'\W+')
corp_re = re.compile('( (group|holding(s)?( co)?|inc(orporated)?|ltd|l ?l? ?[cp]|co(rp(oration)?|mpany)?|s[ae]|plc))+$')
and_re = re.compile(' & ')
punc1_re = re.compile(r'(?<=\S)[\'’´\.](?=\S)')
punc2_re = re.compile(r'[\s\.,:;/\'"`´‘’“”\(\)\[\]\{\}_—\-?$=!]+')

STOPWORDS = nltk.corpus.stopwords.words('english')
STOPWORDS.remove("am")
STOPWORDS.remove("up")
STOPWORDS.remove("in")
STOPWORDS.remove("on")
STOPWORDS.remove("all")
STOPWORDS.remove("any")
STOPWORDS.remove("most")
STOPWORDS.remove("no")
STOPWORDS.remove("nor")
STOPWORDS.remove("own")
STOPWORDS.remove("same")
STOPWORDS.remove("so")
STOPWORDS.remove("very")
STOPWORDS.remove("s")
STOPWORDS.remove("t")
STOPWORDS.remove("d")
STOPWORDS.remove("ll")
STOPWORDS.remove("m")
STOPWORDS.remove("o")
STOPWORDS.remove("re")
STOPWORDS.remove("ve")
STOPWORDS.remove("y")

#compile regex patterns to reuse
STOPWORD_RE = re.compile(r'\b(the|of|and|in|on)\b', re.IGNORECASE)
CORP_SUFFIX_RE = re.compile(r'\b(inc|corp|ltd|llc|plc|co|company|limited)\b', re.IGNORECASE)
PDF_PATTERN_RE = re.compile(r'\s[0-9]*\s[km]b\s*pdf', re.IGNORECASE)
PUNCT_RE = re.compile(r'[^\w\s-]')  # match punctuation
MULTISPACE_RE = re.compile(r'\s+')

stopword_re_str = r""
for word in STOPWORDS:
	stopword_re_str += r'\b' + word + r'\b|'
stopword_re = re.compile(stopword_re_str[:-1]) # The negative 1 is for the fencepost |

NON_FINANCIAL_ORG_TERMS = [
    'university', 'college', 'school', 'institute', 'academy', 
    'hospital', 'medical center', 'health system', 'center', 
    'commission', 'authority', 'association', 'society', 
    'foundation', 'transportation services', 'district', 
    'chamber', 'commerce', 'library', 'museum', 'public', 
    'city', 'county', 'town', 'government', 'state', 'federal',
    'ministry', 'department', 'office'
]

NON_FINANCIAL_RE = re.compile(r'\b(' + '|'.join(NON_FINANCIAL_ORG_TERMS) + r')\b', re.IGNORECASE)

BASE_DIR = "/Users/aawesomez/Documents/UROP/NLP-regextable/"
# BASE_DIR = "/Users/jameschen/Team Name Dropbox/James Chen/JLW-FINREG-PARTICIPATION/"
# BASE_DIR = "/Users/jameschen/Documents/Code/JLW-FINREG-PARTICIPATION/"
# DB_PATH = BASE_DIR + "data/master.sqlite"
DB_PATH = BASE_DIR + "Data/master.sqlite"
# LAST_SAVE_DATASET_DATE = "20210824"
LAST_SAVE_DATASET_DATE = "20220402" # Needs to be set to the last date the 'rebuild datasets' part of this code was run

# Function to calculate longest common substring, from https://www.geeksforgeeks.org/print-longest-common-substring/
# function to find and print 
# the longest common substring of
# X[0..m-1] and Y[0..n-1]
def get_longest_common_substring(X, Y, m, n):
 
    # Create a table to store lengths of
    # longest common suffixes of substrings.
    # Note that LCSuff[i][j] contains length
    # of longest common suffix of X[0..i-1] and
    # Y[0..j-1]. The first row and first
    # column entries have no logical meaning,
    # they are used only for simplicity of program
    LCSuff = [[0 for i in range(n + 1)]
                 for j in range(m + 1)]
 
    # To store length of the
    # longest common substring
    length = 0
 
    # To store the index of the cell
    # which contains the maximum value.
    # This cell's index helps in building
    # up the longest common substring
    # from right to left.
    row, col = 0, 0
 
    # Following steps build LCSuff[m+1][n+1]
    # in bottom up fashion.
    for i in range(m + 1):
        for j in range(n + 1):
            if i == 0 or j == 0:
                LCSuff[i][j] = 0
            elif X[i - 1] == Y[j - 1]:
                LCSuff[i][j] = LCSuff[i - 1][j - 1] + 1
                if length < LCSuff[i][j]:
                    length = LCSuff[i][j]
                    row = i
                    col = j
            else:
                LCSuff[i][j] = 0
 
    # if true, then no common substring exists
    if length == 0:
        return ""
 
    # allocate space for the longest
    # common substring
    resultStr = ['0'] * length
 
    # traverse up diagonally form the
    # (row, col) cell until LCSuff[row][col] != 0
    while LCSuff[row][col] != 0:
        length -= 1
        resultStr[length] = X[row - 1] # or Y[col-1]
 
        # move diagonally up to previous cell
        row -= 1
        col -= 1
 
    # required longest common substring
    longest_common_substring = ''.join(resultStr)

    return longest_common_substring


# Function from Brad Hackinen's NAMA
def basicHash(s):
    '''
    A simple case and puctuation-insensitive hash
    '''
    s = s.lower()
    s = re.sub(and_re,' and ',s)
    s = re.sub(punc1_re,'',s)
    s = re.sub(punc2_re,' ',s)
    s = s.strip()

    return s

# Function from Brad Hackinen's NAMA
def corpHash(s):
    '''
    A hash function for corporate subsidiaries
    Insensitive to
        -case & punctation
        -'the' prefix
        -common corporation suffixes, including 'holding co'
    '''
    s = basicHash(s)
    if s.startswith('the '):
        s = s[4:]

    s = re.sub(corp_re,'',s,count=1)

    return s

# function to clean org names
def clean_fin_org_names(name: str) -> str:
    if name is None or not isinstance(name, str) or name == "NA":
        return ""
    
    # James strip metadata from name
    name = name.split(',')[0]
    #Remove patterns like "10 kb pdf"
    name = PDF_PATTERN_RE.sub("", name)

    #Unicode and punctuation cleanup
    name = corp_simplify_utils.normalize_unicode(name)
    name = PUNCT_RE.sub(" ", name)

    #Remove corporate suffixes and stopwords and non-financial entity
    name = CORP_SUFFIX_RE.sub("", name)
    name = NON_FINANCIAL_RE.sub("", name)
    name = STOPWORD_RE.sub("", name)

    #Normalize spacing and lowercase
    name = MULTISPACE_RE.sub(" ", name).strip().lower()

    return name

In [ ]:
# Locate data directory and read in data files
import os
from pathlib import Path

current_dir = Path(os.getcwd()).parent
data_dir = current_dir / 'data'
data_dir = data_dir.resolve()

try:
    compustat_df = pd.read_csv(data_dir / 'CompustatNames.csv')
    cik_df = pd.read_csv(data_dir / 'CIK.csv')
    fdic_df = pd.read_csv(data_dir / 'FDIC_clean.csv') # Using your 'FDIC_clean.csv'
    sec_df = pd.read_csv(data_dir / 'SEC_Institutions.csv')
except FileNotFoundError as e:
    print(f"Error loading file: {e}")
    print(f"Please make sure your CSV files are in the directory: {data_dir}")
    exit()
    

In [ ]:
# cleaning and standardizing organization names
compustat_df['std_name'] = compustat_df['conm'].apply(clean_fin_org_names)
fdic_df['std_name'] = fdic_df['NAME'].apply(clean_fin_org_names)
sec_df['std_name'] = sec_df['Name'].apply(clean_fin_org_names)
cik_df['std_name'] = cik_df['company_name'].apply(clean_fin_org_names)

In [ ]:
print(cik_df.shape)
cik_df.head(10)

In [ ]:
duplicates_cik_id = cik_df['cik'].duplicated(keep = False)
# duplicates_cik_id.sum()

duplicates_cik = cik_df[duplicates_cik_id]
duplicates_cik
duplicates_cik[duplicates_cik['cik'] == 1137095.0]

In [ ]:
print(sec_df.shape)
sec_df.head(10)

In [ ]:
print(compustat_df.shape)
compustat_df.head(100)

In [ ]:
# cusip is unique 
duplicates_compustat_gvkey = compustat_df['gvkey'].duplicated(keep = False)
# duplicates_compustat_gvkey.sum()

# These are just N/A values, so no need to group by cusip before. 
duplicates_compustat = compustat_df[duplicates_compustat_gvkey]
duplicates_compustat
# All entities are unique in compustat

In [ ]:
print(fdic_df.shape)
fdic_df.head(10)

In [ ]:
# FED_RSSD is unique, and there are duplicates 
duplicates_FDIC_RSSD = fdic_df['FED_RSSD'].duplicated(keep = False)

duplicates_fdic = fdic_df[duplicates_FDIC_RSSD]
duplicates_fdic

In [ ]:
# Test if CIK is already in SEC
sec_ciks = set(sec_df['CIK'].dropna())
cik_ciks = set(cik_df['cik'].dropna())

print(f"Unique CIKs in sec_df: {len(sec_ciks)}")
print(f"Unique CIKs in cik_df: {len(cik_ciks)}")
print(f"Is SEC_Institutions.csv a subset of CIK.csv? {sec_ciks.issubset(cik_ciks)}")

compustat_ciks = set(compustat_df['cik'].dropna())
print(f"CIKs in compustat_df: {len(compustat_ciks)}")
print(f"Is CompustatNames.csv a subset of CIK.csv? {compustat_ciks.issubset(cik_ciks)}")


SEC_Institutions.csv is a subset of CIK.csv --- > Don't need to merge SEC into the crosswalk. 

In [ ]:
# This is the final dataframe crosswalk where we will be merging entities into. 

cols = ['standardized_names', 'aliases', 'cik', 'FED_RSSD', 'sources', 'matching_type', 'fuzzy_matching_score']
final_crosswalk_df = pd.DataFrame(columns = cols)

In [ ]:
# Merging entities in cik_df based on the unique cik id value. 

grouped_by_cik_id = cik_df.groupby('cik')
confident_matches = []
# count = 0
for cik_value, group in grouped_by_cik_id:
    # count += 1
    # if count % 1000 == 0:
    #   print(count)
    
    if len(group) > 1:
        # Aggregate the data based on cik
        new_match_keys = {
            'cik': group['cik'].dropna().unique().tolist(),
            # Now aggregate the std_name to see all variations found for cik
            'standardized_names': '|'.join(group['std_name'].dropna().unique()),
            # Aggregate other fields as before
            'aliases': '|'.join(group['company_name'].dropna().unique()),
            'sources': 'cik',
            'matching_type': 'cik_id_match'
        }
        confident_matches.append(new_match_keys)
    else: 
        unmatched_keys = {
            'cik': group['cik'].dropna().unique().tolist(),
            'standardized_names': group['std_name'].iloc[0],
            'aliases': group['company_name'].iloc[0],
            'sources': 'cik'
        }
        confident_matches.append(unmatched_keys)
         
pd.set_option('display.max_colwidth', None)
enriched_cik_df = pd.DataFrame(confident_matches)
print(f"cik_df reduced to {len(enriched_cik_df)} entities after merging.")

enriched_cik_df

In [ ]:
# Merge the enriched_cik_df into final_crosswalk_df

final_crosswalk_df = pd.concat([final_crosswalk_df, enriched_cik_df])
final_crosswalk_df

In [ ]:
final_crosswalk_df[final_crosswalk_df['matching_type'] == 'cik_id_match']

In [ ]:
# Test if there are cik ids from compustat_df that match final_crosswalk_df
computstat_ciks = set(compustat_df['cik'].dropna())
cik_ciks = set(cik_df['cik'].dropna())

print(f"compustat_df does not have any cik elements in common with final_crosswalk_df: {cik_ciks.isdisjoint(compustat_ciks)}")

In [ ]:
temp_compustat_df = compustat_df[compustat_df['cik'].notna()].copy()

# This function is used to clean the cik values into a string,
# so that the merge function can be properly called. 
def clean_cik(value):
    # If it's a list, take the first element
    if isinstance(value, list):
        if len(value) > 0:
            value = value[0]
        else:
            return ""
    
    # If it's null, return an empty string
    if pd.isna(value) or value == "":
        return ""
    
    # Convert the value to float, then int, then string to remove the '.0'
    try:
        return str(int(float(value)))
    except (ValueError, TypeError):
        return str(value)

# Apply the cleaning function to both dataframes
temp_compustat_df['cik'] = temp_compustat_df['cik'].astype('string')
temp_compustat_df.loc[:, 'cik'] = temp_compustat_df['cik'].apply(clean_cik)
final_crosswalk_df.loc[:, 'cik'] = final_crosswalk_df['cik'].apply(clean_cik)
temp_compustat_df

In [ ]:
# # Merge compustat_df into final_crosswalk_df based on cik id. 
# Create a dataframe of entities that were not merged called remaining_compustat_df 

final_crosswalk_df = final_crosswalk_df.merge(temp_compustat_df, on = 'cik', how = 'left', suffixes=('','_other'), indicator=True)
final_crosswalk_df

In [ ]:
final_crosswalk_df[final_crosswalk_df['_merge'] == 'both']

In [ ]:
# Clean up final_crosswalk_df

mask = final_crosswalk_df['_merge'] == 'both'

# add new std_name
standardized_names = final_crosswalk_df['standardized_names'].astype('string')
new_standardized_name = final_crosswalk_df['std_name'].astype('string')

final_crosswalk_df['standardized_names'] = standardized_names.where(
    new_standardized_name.isna() | (standardized_names == new_standardized_name),
    standardized_names + '|' + new_standardized_name
).fillna(new_standardized_name)

# add new alias
aliases = final_crosswalk_df['aliases'].astype('string')
new_alias = final_crosswalk_df['conm'].astype('string')

final_crosswalk_df['aliases'] = aliases.where(
    new_alias.isna() | (aliases == new_alias),
    aliases + '|' + new_alias
).fillna(new_alias)

# add the source compustat
final_crosswalk_df.loc[mask, 'sources'] = (
    final_crosswalk_df.loc[mask, 'sources']
    .fillna('')
    .where(
        final_crosswalk_df.loc[mask, 'sources'].isna(),
        final_crosswalk_df.loc[mask, 'sources'] + ','
    )
    + 'compustat'
)

# add the entity is matched by cik_id_match
final_crosswalk_df.loc[
    mask,
    'matching_type'
] = 'cik_id_match'

final_crosswalk_df

In [ ]:
print(final_crosswalk_df['_merge'].value_counts())

In [ ]:
# Cleaning up final_crosswalk_df

final_crosswalk_df = final_crosswalk_df.drop(columns=['Unnamed: 0','gvkey', 'conm', 'tic', 'cusip', 'sic', \
                                                      'naics', 'gsubind', 'gind', 'year1', 'year2', 'std_name', '_merge'])

In [ ]:
final_crosswalk_df

In [ ]:
rejected_compustat_df = temp_compustat_df[~temp_compustat_df['cik'].isin(final_crosswalk_df['cik'])].reset_index(drop=True)
rejected_compustat_df


In [ ]:
print(f"The number of compustat entities that didn't match anything: {len(rejected_compustat_df)}")

In [ ]:
rejected_compustat_df = rejected_compustat_df.drop(columns = ['Unnamed: 0', 'gvkey', 'tic', 'cusip', 'sic', 
                                                              'naics', 'gsubind', 'gind', 'year1', 'year2', ])
rejected_compustat_df = rejected_compustat_df.rename(columns={'std_name': 'standardized_names', 'conm': 'aliases'})
rejected_compustat_df['sources'] = 'compustat'

In [ ]:
rejected_compustat_df

In [ ]:
final_crosswalk_df = pd.concat([final_crosswalk_df, rejected_compustat_df], axis = 0, ignore_index=True)

In [ ]:
final_crosswalk_df

In [ ]:
### THIS IS A INEFFICIENT ALGORITHM TO MATCH COMPUSTAT INTO FINAL_CROSSWALK_DF BASED ON CIK

# Merge compustat_df into final_crosswalk_df based on cik id. 
# Create a dataframe of entities that were not merged called remaining_compustat_df 

# New df with the whole row of the rows of compustat with a cik id

# TO DO: Fix to improve time complexity, use merge function of pandas

# temp_compustat_df = compustat_df['cik'].dropna()
# temp_compustat_df = compustat_df.loc[temp_compustat_df.index]
# count = 0 
# for i in range(len(temp_compustat_df)):
#     count += 1
#     if count % 1000 == 0:
#       print(count)
#     for j in range(len(final_crosswalk_df)):
#         current_compustat_row = temp_compustat_df.iloc[i]
#         current_crosswalk_row = final_crosswalk_df.iloc[j]
#         if current_compustat_row['cik'] == current_crosswalk_row['cik']:
#             current_standardized_name = final_crosswalk_df.iloc[j]['standardized_names']
#             final_crosswalk_df.iloc[j, 0] = current_standardized_name + "," + current_compustat_row['std_name']
            
#             current_alias_name = final_crosswalk_df.iloc[j]['aliases']
#             final_crosswalk_df.iloc[j, 1] = current_alias_name + "," + current_compustat_row['conm']

#             current_sources = final_crosswalk_df.iloc[j]['sources']
#             final_crosswalk_df.iloc[j, 4] = current_sources + "," + "compustat"
            
#             current_matching_types = final_crosswalk_df.iloc[j]['matching_type']
#             if isinstance(current_matching_types, float):
#                 # If it's a float (NaN), treat it as an empty string for the check
#                 current_matching_types = ""
#             if 'cik_id' in current_matching_types:
#                 continue
#             final_crosswalk_df.iloc[j, 5] = current_matching_types + "," + "cik_id"        
            

In [ ]:
fdic_df.head()

In [ ]:
# Merge entities in fdic based on FED_RSSD values with itself to remove duplicates. 
# New df called enriched_fdic_df

grouped_by_FED_RSSD = fdic_df.groupby('FED_RSSD')
confident_matches = []
# count = 0
for FED_RSSD_value, group in grouped_by_FED_RSSD:
    # count += 1
    # if count % 1000 == 0:
    #   print(count)
    
    if len(group) > 1:
        # Aggregate the data based on cik
        new_match_keys = {
            'FED_RSSD': group['FED_RSSD'].dropna().unique().tolist(),
            # Now aggregate the std_name to see all variations found for FED_RSSD
            'standardized_names': '|'.join(group['std_name'].dropna().unique()),
            # Aggregate other fields as before
            'aliases': '|'.join(group['NAME'].dropna().unique()),
            'sources': 'fdic',
            'matching_type': 'FED_RSSD_match'
        }
        confident_matches.append(new_match_keys)
    else: 
        unmatched_keys = {
            'FED_RSSD': group['FED_RSSD'].dropna().unique().tolist(),
            'standardized_names': group['std_name'].iloc[0],
            'aliases': group['NAME'].iloc[0],
            'sources': 'fdic'
        }
        confident_matches.append(unmatched_keys)
         
pd.set_option('display.max_colwidth', None)
enriched_fdic_df = pd.DataFrame(confident_matches)
print(f"fdic_df reduced to {len(enriched_cik_df)} entities after merging.")

enriched_fdic_df

In [ ]:
enriched_fdic_df[enriched_fdic_df['matching_type'] == 'FED_RSSD_match']

In [ ]:
# Checking to see which entities in fdic are qualified to be matched 
# based on an exact standardized_name match. Standardized names with more than 
# one appearence in enriched fdic do not qualify to be matched into final_crosswalk_df 
# based on this method, because it is ambiguous as to whether which one matches the entity 
# in final_crosswalk_df when we know they are different because they have different 
# FED_RSSD ids. 

# unqualified_for_standardized_names_matching = enriched_fdic_df['standardized_names'].duplicated(keep=False)
# qualified_for_standardized_names_matching = enriched_fdic_df[~unqualified_for_standardized_names_matching]

In [ ]:
# qualified_for_standardized_names_matching[qualified_for_standardized_names_matching['matching_type'] == 'FED_RSSD_match']

In [ ]:
# Merge enriched_fdic into final_crosswalk_df based on exact standardized_name matching. 

final_crosswalk_df_exploded = (
    final_crosswalk_df.assign(standardized_names = final_crosswalk_df['standardized_names'].str.split('|'))
    .explode('standardized_names')  
)

final_crosswalk_df_exploded['standardized_names'] = (
    final_crosswalk_df_exploded['standardized_names'].str.strip()
)

final_crosswalk_df_exploded

In [ ]:
enriched_fdic_df_exploded = (
    enriched_fdic_df.assign(standardized_names = enriched_fdic_df['standardized_names'].str.split('|'))
    .explode('standardized_names')  
)

enriched_fdic_df_exploded['standardized_names'] = (
    enriched_fdic_df_exploded['standardized_names'].str.strip()
)

enriched_fdic_df_exploded

In [ ]:
unqualified_for_standardized_names_matching_fdic = enriched_fdic_df_exploded['standardized_names'].duplicated(keep=False)
qualified_for_standardized_names_matching__fdic_exploded = enriched_fdic_df_exploded[~unqualified_for_standardized_names_matching_fdic]
qualified_for_standardized_names_matching__fdic_exploded

In [ ]:
unqualified_for_standardized_names_matching_final = final_crosswalk_df_exploded['standardized_names'].duplicated(keep=False)
qualified_for_standardized_names_matching_final_exploded = final_crosswalk_df_exploded[~unqualified_for_standardized_names_matching_final]
qualified_for_standardized_names_matching_final_exploded

In [ ]:
qualified_for_standardized_names_matching_final_exploded['exploded_index'] = qualified_for_standardized_names_matching_final_exploded.index
qualified_for_standardized_names_matching__fdic_exploded['exploded_index'] = qualified_for_standardized_names_matching__fdic_exploded.index

In [ ]:
overlap = final_crosswalk_df.columns.intersection(qualified_for_standardized_names_matching__fdic_exploded.columns)

qualified_for_standardized_names_matching__fdic_exploded = qualified_for_standardized_names_matching__fdic_exploded.rename(
    columns={
        c: f'df2_{c}'
        for c in overlap
        if c != 'standardized_names'
    }
)


In [ ]:
qualified_for_standardized_names_matching__fdic_exploded[qualified_for_standardized_names_matching__fdic_exploded['df2_aliases'] == 'Bank of Hawaii']

In [ ]:

qualified_for_standardized_names_matching_final_exploded[qualified_for_standardized_names_matching_final_exploded['standardized_names'] == 'bank hawaii']

In [ ]:
merged = qualified_for_standardized_names_matching_final_exploded.merge(
    qualified_for_standardized_names_matching__fdic_exploded,
    on='standardized_names',
    how='left',
    suffixes=('', '_df2'),
    indicator=True
)

In [ ]:
merged

In [ ]:
merged[merged['_merge'] == 'both']

In [ ]:
# Clean up merged before turning back into final_crosswalk_df

mask = merged['_merge'] == 'both'

# add new alias
aliases = merged['aliases'].astype('string')
new_alias = merged['df2_aliases'].astype('string')

merged['aliases'] = aliases.where(
    new_alias.isna() | (aliases == new_alias),
    aliases + '|' + new_alias
).fillna(new_alias)

# add the source fdic
merged.loc[mask, 'sources'] = (
    merged.loc[mask, 'sources']
    .fillna('')
    .where(
        merged.loc[mask, 'sources'].isna(),
        merged.loc[mask, 'sources'] + ','
    )
    + 'fdic'
)

merged.loc[mask, 'FED_RSSD'] = (
    merged.loc[mask, 'df2_FED_RSSD']
)

# add the entity is matched by standardized_name_matching
merged.loc[mask, 'matching_type'] = (
    merged.loc[mask, 'matching_type']
    .fillna('')
    .where(
        merged.loc[mask, 'matching_type'].isna(),
        merged.loc[mask, 'matching_type'] + ','
    )
    + merged.loc[mask, 'df2_matching_type'].fillna('') + 'standardized_name_matching'
)

merged

In [ ]:
merged[merged['_merge'] == 'both']

In [ ]:
merged = merged.drop(columns=['df2_FED_RSSD','df2_aliases', 'df2_sources', 'df2_matching_type', 'exploded_index', 'exploded_index_df2', '_merge'])

In [ ]:
merged

In [ ]:
qualified_for_fuzzy_matching = qualified_for_standardized_names_matching__fdic_exploded[\
    ~qualified_for_standardized_names_matching__fdic_exploded['standardized_names'].isin(merged['standardized_names'])].reset_index(drop = True) 
qualified_for_fuzzy_matching

In [ ]:
def normalize_to_list(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return [int(i) for i in x]  # convert to int
    return [int(x)]

In [ ]:
# Create a testing mode to sample data for faster fuzzy matching
TESTING_MODE = True
SAMPLE_FRAC = 0.01 # choose percent of data to sample in testing mode

if TESTING_MODE:
    print(f"--- RUNNING FUZZY MATCHING IN TESTING MODE (Sample: {SAMPLE_FRAC*100}%) ---")
    
    # Take a random sample of your 'unmatched_for_fuzzy' DataFrame
    qualified_for_fuzzy_matching = qualified_for_fuzzy_matching.sample(frac=SAMPLE_FRAC, random_state=42)
    
    print(f"  'qualified_for_fuzzy_matching' sampled to: {len(qualified_for_fuzzy_matching)} rows")
    print("-------------------------------------------------")
    
else:
    print(f"--- RUNNING FUZZY MATCHING IN FULL PRODUCTION MODE ---")
    print(f"  'qualified_for_fuzzy_matching' full size: {len(qualified_for_fuzzy_matching)} rows")
    print("-------------------------------------------------")

In [ ]:
from rapidfuzz import process, fuzz

# Merge qualified_for_fuzzy_matching into merged based on fuzzy matching. 
# 1. Go through each entity in remaining qualified_for_fuzzy_matching and go through the final_crosswalk_df entities
# 2. If the score is higher than the desired threshold, append the values to the existing Series. 

# Prepare list of all names from merged, flattened for multiple names per row
crosswalk_alias_list = []
crosswalk_idx_list = []
merged['FED_RSSD'] = merged['FED_RSSD'].apply(normalize_to_list)
merged['fuzzy_matching_score'] = [[] for _ in range(len(merged))]

print('Beginning string splitting')
count = 0
for idx, row in merged.iterrows():
    count += 1 
    if count % 1000 == 0:
        print(count)
    names = row['aliases'].split('|')
    for name in names:
        crosswalk_alias_list.append(name)
        crosswalk_idx_list.append(idx)

# Now go through qualified_for_fuzzy_matching
count = 0
print('beginning matching')
for i, new_row in qualified_for_fuzzy_matching.iterrows():
    count += 1 
    if count % 50 == 0:
        print(count)
    
    new_alias = new_row['df2_aliases']
    # Find best matches with threshold 90
    matches = process.extract(
        new_alias,
        crosswalk_alias_list,
        scorer=fuzz.token_set_ratio,
        score_cutoff=50
    )
    
    for match_name, score, match_pos in matches:
        idx = crosswalk_idx_list[match_pos]
        
        # Skip merge if 'fdic' already in sources
        existing_sources = merged.at[idx, 'sources']
        if existing_sources and 'fdic' in existing_sources.split(','):
            continue

        # Update merged row
        merged.at[idx, 'aliases'] = (
            merged.at[idx, 'aliases'] + '|' + new_alias
        )
        mask = qualified_for_fuzzy_matching['df2_aliases'] == new_alias
        alias_val = qualified_for_fuzzy_matching.loc[mask, 'standardized_names'].iloc[0]
        merged.at[idx, 'standardized_names'] = (
            merged.at[idx, 'standardized_names'] + '|' + alias_val
        )
        rssd_val = qualified_for_fuzzy_matching.loc[mask, 'df2_FED_RSSD'].iloc[0]
        # If rssd_val is a list, get the first element
        if isinstance(rssd_val, list) and len(rssd_val) > 0:
            rssd_val = rssd_val[0]
        merged.at[idx, 'FED_RSSD'].append(int(rssd_val))

        val_source = merged.at[idx, 'sources']
        merged.at[idx, 'sources'] = (
            ("" if pd.isna(val_source) else val_source + ',') + 'fdic'
        )
        val_matching_type = merged.at[idx, 'matching_type']
        merged.at[idx, 'matching_type'] = (
            ("" if pd.isna(val_matching_type) else val_matching_type + ',') + 'fuzzy_matching'
        )
        # Append fuzzy score to the column
        merged.at[idx, 'fuzzy_matching_score'].append(score)


In [ ]:
fuzzy_rows = merged[merged['matching_type'].str.contains('fuzzy_matching', na=False)]
fuzzy_rows

In [ ]:
enriched_fdic_df['FED_RSSD'] = (
    enriched_fdic_df['FED_RSSD']
    .str[0]
    .astype('Int64')   
)

In [ ]:
merged['FED_RSSD'] = (
    merged['FED_RSSD']
    .str[0]
    .astype('Int64')  
)

In [ ]:

remaining_fdic_df = enriched_fdic_df[~enriched_fdic_df['FED_RSSD'].isin(merged['FED_RSSD'])]
final_crosswalk_df = pd.concat([merged, remaining_fdic_df], ignore_index=True)

In [ ]:
enriched_fdic_df

In [ ]:
merged[merged['FED_RSSD'].notna()]

In [ ]:
remaining_fdic_df

In [ ]:
enriched_fdic_df

In [ ]:
final_crosswalk_df

In [ ]:
# # Imported function from the regextable-python repository
# # get_match_candidate_score get the matching score between two names
# def get_match_candidate_score(frequency_dict, org_name, candidate_match_name):
#     if not isinstance(org_name, str):
#         org_name = ""
#     if not isinstance(candidate_match_name, str):
#         candidate_match_name = ""
    
#     if not org_name or not candidate_match_name:
#         return 0.0
    
#     org_tokens = org_name.split(' ')
    
#     # tokenize the candidate match
#     candidate_match_tokens = set(candidate_match_name.split(" "))

#     max_dist = 1

#     # Calculate the match score
#     total_inverse_frequency = 0
#     total_matching_inverse_frequency = 0
#     tokenized_name = org_tokens
#     for token in tokenized_name:
#         token_frequency = frequency_dict.get(token, 999999) # if token not found, give high frequency to ignore it
#         token_inverse_frequency = 1.0/token_frequency
#         total_inverse_frequency += token_inverse_frequency

#         best_token_similarity = 0.0
        
#         for candidate_token in candidate_match_tokens:
#             if not token or not candidate_token:
#                 continue

#             dist = damerau_levenshtein_distance(token, candidate_token)

#             if dist <= max_dist:
#                 max_len = max(len(token), len(candidate_token))
#                 if max_len == 0: continue
#                 similarity = 1.0 - (dist / max_len)

#                 best_token_similarity = max(best_token_similarity, similarity)
#         if best_token_similarity > 0.0:
#             total_matching_inverse_frequency += token_inverse_frequency * best_token_similarity
    
#     try:
#         match_score = total_matching_inverse_frequency / total_inverse_frequency
#     except ZeroDivisionError:
#         match_score = 0.0
    
#     #Multiplicative DL Penalty
#     m = len(org_name)
#     n = len(candidate_match_name)
#     if m == 0 or n == 0: return 0.0

#     dl_distance = damerau_levenshtein_distance(org_name, candidate_match_name)
#     normalized_dl = dl_distance / max(m, n)

#     final_score = match_score * (1 - normalized_dl)
#     return max(0.0, final_score)
        
        
# def clean_match_score(x):
#     if x is None or x is np.nan or pd.isnull(x) or x == "":
#         return np.nan
#     elif isinstance(x, str) and not x.isnumeric():
#         unit_multiplier = 1
#         if "B" in x:
#             x = x[:-1]
#             unit_multiplier = 1000000000
#         if "M" in x:
#             x = x[:-1]
#             unit_multiplier = 1000000
#         if "K" in x:
#             x = x[:-1]
#             unit_multiplier = 1000
#         x = x.replace(",", "")
#         try:
#             x = float(x) * unit_multiplier
#             return x
#         except:
#             return np.nan
#     else:
#         return float(x)

# from collections import defaultdict

# frequency_df = pd.DataFrame()
# frequency_df['std_name'] = unmatched_for_fuzzy['std_name']
# frequency_dict = defaultdict(int)

# # Iterate over the clean organization names
# for name in frequency_df['std_name']:
#     # Split the name into tokens (words)
#     tokens = name.split(' ')

#     # Update the count for each token
#     for token in tokens:
#         # Filter out empty strings that might result from extra spaces
#         if token:
#             frequency_dict[token] += 1

# # Convert defaultdict back to a standard dict for the function
# frequency_data = dict(frequency_dict)

# print("Printing the frequency_dict")
# print(frequency_data)
# from union_find import UnionFind

# # initialize union-find here
# uf = UnionFind(len(unmatched_for_fuzzy)) 
# print(len(uf.parent))

# # uf.unite(1, 1000)
# # uf.unite(1, 30)
# # set_i = uf.find(1000)
# # set_j = uf.find(1)
# # set_k = uf.find(30)

# # print(f'set_i: {set_i}, set_j: {set_j}, set_k: {set_k}')
# # print(uf.find(200))
# from rapidfuzz import process, fuzz
# from rapidfuzz.distance import JaroWinkler
# # score = fuzz.token_set_ratio("bank of american", 'bank of america')
# # score

# # score = JaroWinkler.similarity("bank of american", "bank of amercia")
# # score
# # count = 0
# for i in range(len(unmatched_for_fuzzy)):
#    # count += 1 
#    # if count % 10 == 0:
#    #    print(count)
      
#    item_i = unmatched_for_fuzzy.iloc[i]
#    for j in range(i+1, len(unmatched_for_fuzzy)):
#       item_j = unmatched_for_fuzzy.iloc[j]
      
#       # add union-find here
#       set_i = uf.find(i)
#       set_j = uf.find(j)
#       # When two index are in the same set, they already matched, skip
#       if set_i == set_j:
#          continue 
      
#       # score = get_match_candidate_score(frequency_dict, item_i['std_name'], item_j['std_name']) * 100
#       score = fuzz.token_set_ratio(item_i['std_name'], item_j['std_name'])
#       # score = JaroWinkler.similarity(item_i['std_name'], item_j['std_name']) * 100
#       # if j % 3 == 0: 
#       #    print(f"i is {i} and j is {j}, score is {score}")
      
#       if score < 93: 
#          # print(f"Warn — itemj name: {item_j['std_name']}, itemi name: {item_i['std_name']}")
#          continue
      
#       uf.unite(i, j)
#       # print(f"Match: i is {i} and j is {j}")
#       # print(f"item_i: {item_i['std_name']}, item_j: {item_j['std_name']}")
      
      
#    if i== 50:
#       break

# # The dictionary of parents and their row lists so that indices in the same list will be turned into a series
# # that will be merged into the new matched_df
# index_dict = defaultdict(list)
# for i in range(len(uf.parent)):
#     current_parent = uf.find(i)
#     index_dict[current_parent].append(i)
# index_dict = dict(index_dict)
# print(index_dict)
# print(len(index_dict))


# # TODO: Implement the for-loop that goes through all the keys in the sets and appends a new series to matched_df
# cols = ['standardized_names', 'aliases', 'cik', 'sources', 'original_index', 'score']
# matched_df = pd.DataFrame(columns = cols)

# count = 0
# for key in index_dict:
#     count += 1 
#     if count % 1000 == 0:
#         print(count)
#     row_list = index_dict[key]
#     new_std_name = ""
#     new_aliases = ""
#     new_sources = ""
#     new_cik = []
#     new_original_indices = []
    
#     for i in range(len(row_list)):
#         row = row_list[i]
#         if i == len(row_list) - 1:
#             new_std_name += unmatched_for_fuzzy.iloc[row]['std_name']
#             new_aliases += unmatched_for_fuzzy.iloc[row]['raw_name']
#             new_sources += unmatched_for_fuzzy.iloc[row]['source']
#             new_cik.append(unmatched_for_fuzzy.iloc[row]['cik'])
#             new_original_indices.append(row)
#         else:
#             new_std_name += unmatched_for_fuzzy.iloc[row]['std_name'] + "|"
#             new_aliases += unmatched_for_fuzzy.iloc[row]['raw_name'] + "|"
#             new_sources += unmatched_for_fuzzy.iloc[row]['source'] + ","
#             new_cik.append(unmatched_for_fuzzy.iloc[row]['cik']) 
#             new_original_indices.append(row)  
        
#         combined_row_dict = {"standardized_names": new_std_name,
#                              "aliases": new_aliases,
#                              "sources": new_sources,
#                              "cik": new_cik,
#                              "original_indices": new_original_indices} 
#         new_row_series = pd.Series(combined_row_dict)
#         matched_df = pd.concat([matched_df, new_row_series.to_frame().T])
        
# matched_df
          